# Binoculars — Zero-Shot Detection of LLM-Generated Text

Hans et al., **ICML 2024** · [proceedings.mlr.press/v235/hans24a](https://proceedings.mlr.press/v235/hans24a.html) ·
code gốc: https://github.com/ahans30/Binoculars

> **Phạm vi**: chỉ inference + eval theo đúng bộ metric của nhánh supervised
> (AUROC / TPR@FPR / FPR@τ trên `val` + `test_ood_model` + `test_fp`). **Chưa** eval dưới attack —
> xem `docs/attack_eval_plan.md`. Nhánh **Thống kê/Zero-Shot**, **không train**, chỉ 2 LLM frozen.

## ✅ Cặp model: KHỚP ĐÚNG PAPER

`tiiuae/falcon-7b` (observer) + `tiiuae/falcon-7b-instruct` (performer) — đúng mặc định của
`Binoculars.__init__`. **Không còn deviation về backbone.**

Layout GPU cũng khớp code gốc: `detector.py` đặt `DEVICE_1 = cuda:0` cho observer và
`DEVICE_2 = cuda:1` cho performer, **một process, model-parallel**. Notebook này làm y hệt — khác
bản 1.5B trước đây (data-sharding 2 rank), vì 2×13.44 GiB không thể nằm chung một T4.

| | fp16/model | 2 model / 1 GPU (sharding) | 1 model / GPU (model-parallel) |
|---|---|---|---|
| `falcon-7b` ×2 | **13.44 GiB** | 26.9 GiB ❌ | **13.44 GiB ✅** (T4 khả dụng ~14.6 GiB) |
| `Qwen2.5-7B` ×2 | 14.19 GiB | ❌ | 14.19 GiB ⚠️ gần chắc OOM |

Falcon còn có vocab **65.024** (Qwen 152.064) ⇒ tensor logits nhẹ hơn 2,3× — thêm một lý do
kỹ thuật để bám đúng cặp của paper.

## Cơ chế — toàn bộ ở **log-space**, đúng Eq. (1)–(4)

```
B(x)                  = log PPL_{M_perf}(x) / log X-PPL_{M_obs, M_perf}(x)
log PPL_M(x)          = -1/L Σ_i log M(x)_{i, x_i}                       — Eq. (2)
log X-PPL_{M1,M2}(x)  = -1/L Σ_i M1(x)_i · log( M2(x)_i )                — Eq. (3), dot product trên vocab
```

### ⚠️ Tử số dùng model nào — chỗ dễ sai nhất

Eq. (4) viết ký hiệu `log PPL_{M1}`, nhưng **số liệu thật của paper và code gốc đều dùng PERFORMER
(instruct) ở tử số**:

- Table 5 (Appendix A.4) đặt tên cột `PPL (Falcon 7B Instruct)` và `Cross PPL (Falcon 7B, Falcon 7B Instruct)`.
  Kiểm lại: US Constitution `0.6680 / 0.8789 = 0.7601` → đúng score `0.76` paper báo;
  "I have a dream" `1.0000 / 1.2344 = 0.8101` → đúng score paper báo.
- `binoculars/detector.py`: `ppl = perplexity(encodings, performer_logits)`.

Nên: **tử số = log-PPL của performer (shift 1 token)**, **mẫu số = cross-entropy target là phân phối
observer, log-prob của performer, KHÔNG shift, trên toàn bộ `L` vị trí** (đúng `entropy()` gốc).
**Không `exp()`** ở đâu cả — thêm `exp()` là ra một tỉ số khác, không phải Binoculars.

**Quy ước dấu (đăng ký trước):** paper báo *B thấp = AI*; ta lưu `score = -B` để khớp "score cao = AI".
Theo `docs/statistical_branch_plan.md` §4, **không tự lật dấu theo nhãn `val`** — `AUROC(val) < 0.5`
chỉ phát cảnh báo.

## 🎁 Bonus chỉ có được khi khớp đúng cặp paper

Vì backbone y hệt paper, hai **ngưỡng toàn cục gốc** trở nên áp dụng được và notebook báo thêm:

```
BINOCULARS_ACCURACY_THRESHOLD = 0.9015310749276843   # tối ưu F1
BINOCULARS_FPR_THRESHOLD      = 0.8536432310785527   # tối ưu low-FPR (chọn tại 0.01%)
```

Đây là "Binoculars mua về dùng ngay, zero-config" trên dữ liệu của ta — một cột rất đắt cho KLTN.
**Không** thay τ: cột dùng để so với supervised/AL vẫn là τ calibrate trên `val` @ FPR=1%.

## Deviation so với paper

| # | Paper / code gốc | Ta dùng | Tầng | Lý do |
|---|---|---|---|---|
| 1 | `falcon-7b` / `falcon-7b-instruct`, `max_token_observed=512` | **khớp đúng** | — | Không deviation |
| 2 | Ngưỡng toàn cục `0.9015` / `0.8536` để ra nhãn cuối | τ calibrate trên `val` @ FPR=1% **cho cột so sánh chính**; ngưỡng gốc báo song song | bắt buộc | Ngưỡng gốc tune trên dataset của paper, không phải dataset arXiv của ta; supervised/AL cũng calibrate trên `val` → không calibrate thì bảng không công bằng |
| 3 | `use_bfloat16=True` | fp16 cho trọng số; `log_softmax`/tích/tổng ép **fp32** | hạ tầng | T4 (Turing) không có bf16. Worker raise ngay nếu score ra NaN/Inf |
| 4 | `trust_remote_code=True` (code trong repo HF) | `FalconForCausalLM` native của `transformers` | — | Cùng kiến trúc, bản native được maintain; tránh chạy code tuỳ ý từ hub |

## So sánh công bằng với `supervised_models/` và `adversarial_models/`

Giữ **y hệt** 3 nhánh: parquet đầu vào · `melt_pairs`/`melt_fp` · 3 split `val`/`test_ood_model`
(full, inductive)/`test_fp` · `MAX_LEN=512` · `tpr_at_fpr`/`auroc`/`report_split` · `TARGET_FPR=0.01`,
τ từ `val` · schema `metrics.json` + `pred_{split}.parquet`.

Ghi chú khi lên bảng `compare_all`: `n_seeds = 1`, `deterministic = true` (training-free, không có
±std); và `test_ood_model` **không thực sự "OOD"** với detector không train — diễn giải riêng.


In [ ]:
!pip install -q -U "transformers>=4.45" accelerate sentencepiece

In [ ]:
%%writefile score_worker.py
import os, json, random, warnings, time, math, hashlib, gc
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from pathlib import Path

# 2 checkpoint Falcon ~29 GB: day cache sang /kaggle/temp (scratch) de khong an quota output 20 GB.
if Path("/kaggle/temp").exists():
    os.environ.setdefault("HF_HOME", "/kaggle/temp/hf_cache")

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# ── Bản dataset dùng để train/eval ────────────────────────────────────
#   "deconf2"    = 02c_deconfound_v2.ipynb  <- ĐANG DÙNG
#                  canonicalize + truncate_pair_sentences + lọc notation_gap 85%.
#                  shortcut_audit.py đo đủ 22 đặc trưng: tầng process 0.5055 (cổng
#                  cứng, PASS) · tổng 0.8909. Train 7.025 cặp (85% của 8.265).
#   "deconf"     = 02b_deconfound.ipynb. ĐỪNG dùng cho nhánh này. Hai lý do:
#                  (a) con số "surface-GBM 0.6867, G6 PASS" mà 02b tự báo chỉ đo 6
#                      đặc trưng, thiếu avg_word_len (0.7795) lẫn type_token_ratio
#                      (0.7727). Đo đủ 22 thì là 0.9114 -> WARN, không PASS.
#                  (b) truncate_pair cắt thô đúng từ thứ min(len) nên chỉ 52,4% văn
#                      bản còn kết thúc đúng dấu câu (trước khi cắt: 99,0%). Detector
#                      ở file này chấm điểm bằng log-probability của từng token, nên
#                      token bị chặt giữa câu là perplexity GIẢ — và nó BẤT ĐỐI XỨNG,
#                      vì câu AI dài hơn (22,4 vs 20,4 từ) nên bên AI mất nhiều từ
#                      hơn một cách có hệ thống.
#   "normalized" = bản GỐC, CÒN shortcut: chỉ ĐẾM SỐ TỪ đã đoán đúng human/AI ở
#                  AUROC 0.7193 (surface-GBM 0.8080). G6 FAIL.
#
# PROVENANCE của số cũ (REPORT_2026-08-01.md §5) — đã truy được, ĐỪNG suy diễn lại:
# log còn lưu trong output notebook ghi sha256 val=c9d3957ae20b ood=2e9a47ca22e0
# fp=584cb43da181, khớp ĐÚNG dataset_{val,test_ood_model,test_fp}_normalized.parquet.
# Tức mọi số static/detectgpt trong báo cáo đó chạy trên "normalized" — chưa
# canonicalize, chưa cắt cặp — KHÔNG phải "deconf" như dòng tag cũ ghi (dòng đó nằm
# trên %%writefile nên chưa bao giờ tới được worker).
# Hệ quả: AUROC(val) < 0.5 của Fast-DetectGPT (0.4760) và Binoculars (0.4650) KHÔNG
# do tiền xử lý. Nguyên nhân phải tìm ở chỗ khác — mismatch thời đại backbone (gpt2-xl
# 2019 / gpt-j 2021 / falcon 2023 vs generator 2026), hoặc bug implementation. PHẢI có
# positive control trên một tập đã biết kết quả công bố trước khi tin bảng so sánh.
#
# PHẢI trùng tag với supervised_models/ và adversarial_models/ — cả 8 notebook đó đã
# chạy trên "deconf2". Lệch tag thì hai bảng số KHÔNG so được với nhau: deconf 8.265
# cặp vs deconf2 7.025 cặp, và nội dung text cũng khác (cắt theo từ vs cắt biên câu).
#
# PHẢI đặt TRONG body của %%writefile, KHÔNG phải ở đầu cell notebook. Đặt ở đầu cell
# thì %%writefile bị đẩy khỏi dòng 0 -> Jupyter hiểu thành line magic và báo "Line
# magic function `%%writefile` not found"; mà kể cả chạy được thì worker vẫn NameError
# vì nó là tiến trình riêng, không thấy biến của notebook.
#
# LƯU Ý: Kaggle dataset phải chứa dataset_{train,val,test_ood_model,test_fp}_deconf2.parquet
DATASET_TAG = "deconf2"
# ── CONFIG — khop mac dinh cua Binoculars.__init__ ───────────────────────────
IMPL_VERSION = "paper-exact-falcon-pair-modelparallel-v1-2026-07-28"
OBSERVER_MODEL = "tiiuae/falcon-7b"            # M_obs, code goc: DEVICE_1
PERFORMER_MODEL = "tiiuae/falcon-7b-instruct"  # M_perf, code goc: DEVICE_2
MAX_LEN = 512          # = max_token_observed mac dinh
BATCH_SIZE = 1         # an toan VRAM cho 7B fp16 tren T4; thu 2 sau khi SMOKE on dinh
POS_CHUNK = 64         # so vi tri token / luot -> khong giu [B,512,65024] fp32 mot cuc
CHECKPOINT_ROWS = 100
RUN_NAME = "binoculars_falcon7b"
MODEL_DESC = f"{OBSERVER_MODEL} + {PERFORMER_MODEL} (observer/performer)"
FORMULA = "ppl=logPPL_performer(shift1) / xppl=H(p=observer,q=performer,no-shift,all-L); score=-B"
RUN_SIGNATURE = "|".join([IMPL_VERSION, OBSERVER_MODEL, PERFORMER_MODEL, str(MAX_LEN),
                          "fp16-weights-fp32-statistics", FORMULA])

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Can 2 GPU: falcon-7b tren cuda:0, falcon-7b-instruct tren cuda:1. "
                       "1 GPU khong du 2x13.44 GiB.")
OBSERVER_DEVICE = torch.device("cuda:0")
PERFORMER_DEVICE = torch.device("cuda:1")
SMOKE_TEST = os.environ.get("SMOKE", "0") == "1"
SMOKE_PER_CLASS = 16
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def log(*a):
    print("[binoculars]", *a, flush=True)

from transformers import AutoTokenizer, AutoModelForCausalLM

# Paper footnote 2 + assert_tokenizer_consistency() cua code goc: 2 model PHAI chung tokenizer.
obs_tok = AutoTokenizer.from_pretrained(OBSERVER_MODEL)
prf_tok = AutoTokenizer.from_pretrained(PERFORMER_MODEL)
for _t in (obs_tok, prf_tok):
    if _t.pad_token is None:
        _t.pad_token = _t.eos_token
    _t.padding_side = "right"   # causal LM + forward thuong: right-padding moi giu dung position id
if obs_tok.get_vocab() != prf_tok.get_vocab():
    raise RuntimeError("Tokenizer cua observer va performer khac nhau - Binoculars khong dinh nghia duoc.")

# Load tuan tu, low_cpu_mem_usage -> giam dinh RAM host (moi model ~14.4 GB tren dia).
observer = AutoModelForCausalLM.from_pretrained(
    OBSERVER_MODEL, torch_dtype=torch.float16, low_cpu_mem_usage=True).to(OBSERVER_DEVICE).eval()
observer.config.use_cache = False
gc.collect(); torch.cuda.empty_cache()
performer = AutoModelForCausalLM.from_pretrained(
    PERFORMER_MODEL, torch_dtype=torch.float16, low_cpu_mem_usage=True).to(PERFORMER_DEVICE).eval()
performer.config.use_cache = False
gc.collect(); torch.cuda.empty_cache()
log("loaded:", OBSERVER_MODEL, "on", OBSERVER_DEVICE, "|", PERFORMER_MODEL, "on", PERFORMER_DEVICE)
log("impl:", IMPL_VERSION)
for _d in (0, 1):
    log(f"VRAM sau khi load cuda:{_d} = {torch.cuda.memory_allocated(_d)/1024**3:.2f} GiB")

@torch.inference_mode()
def score(texts, batch_size=BATCH_SIZE):
    """score cao = AI.

    Khop reference implementation (ahans30/Binoculars, binoculars/{detector,metrics}.py):
      ppl   = perplexity(encodings, PERFORMER_logits)   -> shift 1 token, mean CE co attention-mask
      x_ppl = entropy(p=OBSERVER, q=PERFORMER)          -> KHONG shift, mean tren toan bo L vi tri
      B     = ppl / x_ppl        (paper: B thap = AI)   -> tra ve -B de "cao = AI"
    Tat ca o log-space (mean NLL), khong exp() o dau ca.
    """
    out, n_short = [], 0
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        # tokenize 2 lan -> 2 BatchEncoding doc lap (BatchEncoding.to() sua in-place, khong clone!)
        enc_obs = obs_tok(batch, return_tensors="pt", truncation=True, max_length=MAX_LEN,
                          padding=True, return_token_type_ids=False)
        enc_prf = prf_tok(batch, return_tensors="pt", truncation=True, max_length=MAX_LEN,
                          padding=True, return_token_type_ids=False)
        if not torch.equal(enc_obs["input_ids"], enc_prf["input_ids"]):
            raise ValueError("2 tokenizer cho ra token IDs khac nhau.")
        enc_obs = enc_obs.to(OBSERVER_DEVICE)
        enc_prf = enc_prf.to(PERFORMER_DEVICE)
        ids = enc_prf["input_ids"]
        am = enc_prf["attention_mask"].float()
        B, L = ids.shape
        n_short += int((am.sum(1) < 64).sum().item())

        # Launch CA HAI forward truoc khi cham vao ket qua -> 2 T4 chay chong nhau (kernel async).
        logits_obs_g0 = observer(**enc_obs).logits
        logits_prf = performer(**enc_prf).logits

        # Code goc crop ve min vocab khi 2 LM co output embedding padding khac nhau (falcon: 65024 ca hai).
        cv = min(logits_obs_g0.size(-1), logits_prf.size(-1))
        if int(ids.max().item()) >= cv:
            raise ValueError(f"Token ID vuot common vocab = {cv}.")
        logits_prf = logits_prf[..., :cv]

        ppl_sum = torch.zeros(B, device=PERFORMER_DEVICE, dtype=torch.float32)
        ce_sum = torch.zeros(B, device=PERFORMER_DEVICE, dtype=torch.float32)
        for a in range(0, L, POS_CHUNK):
            e = min(a + POS_CHUNK, L)
            # chuyen tung chunk logits observer sang GPU cua performer (~8 MB voi chunk 64)
            lp_obs = torch.log_softmax(
                logits_obs_g0[:, a:e, :cv].to(PERFORMER_DEVICE, non_blocking=True).float(), dim=-1)
            lp_prf = torch.log_softmax(logits_prf[:, a:e].float(), dim=-1)

            # mau so - Eq. (3): -sum_v M_obs(v) * log M_perf(v), tren MOI vi tri (khong shift)
            ce_sum += (-(lp_obs.exp() * lp_prf).sum(-1) * am[:, a:e]).sum(1)

            # tu so - Eq. (2): log-PPL cua PERFORMER, logit tai vi tri p du doan token p+1
            pe = min(e, L - 1)
            if pe > a:
                lab = ids[:, a + 1:pe + 1]
                msk = am[:, a + 1:pe + 1]
                tok_lp = lp_prf[:, :pe - a].gather(-1, lab.unsqueeze(-1)).squeeze(-1)
                ppl_sum += -(tok_lp * msk).sum(1)
            del lp_obs, lp_prf
        del logits_obs_g0, logits_prf

        ppl = ppl_sum / am[:, 1:].sum(1).clamp(min=1)     # shift -> mau la L-1 token thuc
        xppl = ce_sum / am.sum(1).clamp(min=1)            # khong shift -> mau la L token thuc
        b = ppl / xppl.clamp(min=1e-9)
        if not torch.isfinite(b).all():
            raise FloatingPointError("Binoculars sinh score NaN/Inf (fp16 overflow?).")
        out.extend((-b).float().cpu().tolist())
    if n_short:
        log(f"[!] {n_short} doc < 64 token - paper (Fig. 2) chi cho thay on dinh tu ~50-100 token tro len")
    return np.array(out)

# ── Data + checkpoint theo chunk; mot process dieu khien ca hai GPU ──────────
KAGGLE_INPUT = "/kaggle/input/datasets/<kaggle-user>/halfdatadetector"

def find_data_dir():
    target = f"dataset_val_{DATASET_TAG}.parquet"
    if (Path(KAGGLE_INPUT) / target).exists():
        return Path(KAGGLE_INPUT)
    for base in [Path(KAGGLE_INPUT), Path("output"), Path("."), Path("/kaggle/input")]:
        if base.exists():
            hits = list(base.rglob(target))
            if hits:
                return hits[0].parent
    raise FileNotFoundError(f"Khong tim thay {target}. Kiem tra KAGGLE_INPUT = {KAGGLE_INPUT}")

DATA_DIR = find_data_dir()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_DIR = OUT_DIR / f"artifacts_{RUN_NAME}"
CHUNK_DIR = OUT_DIR / "chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
log("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR)

def sha256_file(path, block_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

# melt_pairs/melt_fp: GIU Y HET supervised_models/ va adversarial_models/ (bo cot sample_weight
# vi khong train) -> cung tap dong, cung nhan, moi so sanh duoc.
def melt_pairs(df):
    h = pd.DataFrame({"text": df["human_text"], "label": 0, "paper_id": df["paper_id"],
                       "ai_degree": 0, "generator_family": "human"})
    a = pd.DataFrame({"text": df["ai_text"], "label": 1, "paper_id": df["paper_id"],
                       "ai_degree": df.get("ai_degree", 3),
                       "generator_family": df.get("generator_family", "unknown")})
    out = pd.concat([h, a], ignore_index=True)
    return out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)].reset_index(drop=True)

def melt_fp(df):
    has_ai = df["ai_text"].notna() if "ai_text" in df.columns else pd.Series(False, index=df.index)
    text = df["ai_text"].where(has_ai, df["human_text"]) if "ai_text" in df.columns else df["human_text"]
    out = pd.DataFrame({"text": text, "label": 0, "paper_id": df["paper_id"],
                         "ai_degree": df.get("ai_degree", 0).fillna(0).astype(int) if "ai_degree" in df.columns else 0,
                         "generator_family": "human"})
    return out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)].reset_index(drop=True)

def load(name, fp_mode=False):
    p = DATA_DIR / f"dataset_{name}_{DATASET_TAG}.parquet"
    if not p.exists():
        log(f"thieu {p.name} - bo qua"); return None
    df = pd.read_parquet(p)
    return melt_fp(df) if fp_mode else melt_pairs(df)

def maybe_smoke(df):
    if df is None or not SMOKE_TEST:
        return df
    n = min(SMOKE_PER_CLASS, int(df["label"].value_counts().min()))
    return (df.groupby("label", group_keys=False)
              .sample(n=n, random_state=SEED)
              .sample(frac=1, random_state=SEED)
              .reset_index(drop=True))

CHUNK_COLS = ["_row_id", "_run_signature", "_data_sha256", "_expected_n", "_chunk_idx",
              "paper_id", "label", "ai_degree", "generator_family", "score"]

def validate_chunk(part, path, data_sha, expected_n, chunk_idx, start, end):
    missing = set(CHUNK_COLS).difference(part.columns)
    ok = (not missing and set(part["_run_signature"].unique()) == {RUN_SIGNATURE}
          and set(part["_data_sha256"].unique()) == {data_sha}
          and set(part["_expected_n"].unique()) == {expected_n}
          and set(part["_chunk_idx"].unique()) == {chunk_idx}
          and part["_row_id"].tolist() == list(range(start, end))
          and np.isfinite(part["score"].to_numpy()).all())
    if not ok:
        raise RuntimeError(f"Checkpoint cu/hong: {path}. Doi CONFIG thi phai xoa folder "
                           f"{CHUNK_DIR} truoc khi chay lai.")

def score_split(split_name, fp_mode=False):
    data_path = DATA_DIR / f"dataset_{split_name}_{DATASET_TAG}.parquet"
    df = maybe_smoke(load(split_name, fp_mode=fp_mode))
    if df is None or len(df) == 0:
        return
    df["_row_id"] = np.arange(len(df), dtype=np.int64)
    if SMOKE_TEST:
        t0 = time.time()
        s = score(df["text"].tolist())
        log(f"[SMOKE] {split_name}: {len(df)} dong, {time.time()-t0:.1f}s, "
            f"{len(df)/max(time.time()-t0,1e-9):.2f} samples/s, finite={np.isfinite(s).all()}, "
            f"score[min/max]={s.min():.4f}/{s.max():.4f}; KHONG luu checkpoint.")
        return

    data_sha = sha256_file(data_path)
    expected_n = len(df)
    n_chunks = math.ceil(expected_n / CHECKPOINT_ROWS)
    log(f"{split_name}: {expected_n} dong, {n_chunks} chunks, sha256={data_sha[:12]}...")
    split_t0 = time.time(); newly = 0
    for chunk_idx in range(n_chunks):
        start = chunk_idx * CHECKPOINT_ROWS
        end = min(start + CHECKPOINT_ROWS, expected_n)
        out_path = CHUNK_DIR / f"{split_name}_chunk{chunk_idx:05d}.parquet"
        if out_path.exists():
            validate_chunk(pd.read_parquet(out_path), out_path, data_sha, expected_n,
                           chunk_idx, start, end)
            log(f"resume {split_name} chunk {chunk_idx+1}/{n_chunks}: da co")
            continue
        part = df.iloc[start:end].copy()
        t0 = time.time()
        part["score"] = score(part["text"].tolist())
        part["_run_signature"] = RUN_SIGNATURE
        part["_data_sha256"] = data_sha
        part["_expected_n"] = expected_n
        part["_chunk_idx"] = chunk_idx
        tmp = out_path.with_suffix(".tmp.parquet")
        part[CHUNK_COLS].to_parquet(tmp, index=False)
        os.replace(tmp, out_path)   # atomic: session ngat khong de lai checkpoint nua voi
        newly += len(part)
        rate = len(part) / max(time.time() - t0, 1e-9)
        log(f"saved {split_name} chunk {chunk_idx+1}/{n_chunks}: {rate:.2f} samples/s, "
            f"ETA con lai ~{(expected_n-end)/max(rate,1e-9)/3600:.2f}h")
    log(f"XONG {split_name}: moi cham {newly} dong, wall={time.time()-split_t0:.1f}s")

for _split, _fp in [("val", False), ("test_ood_model", False), ("test_fp", True)]:
    score_split(_split, fp_mode=_fp)

for _d in (0, 1):
    log(f"peak VRAM cuda:{_d} = {torch.cuda.max_memory_allocated(_d)/1024**3:.2f} GiB")
log("XONG SMOKE worker." if SMOKE_TEST else "XONG FULL worker.")

# ═══ CHẾ ĐỘ MIX — chấm tập trộn của 06_mixing để dựng đường dose-response ═════════
# Bật bằng: RUN_MIX=1 <lệnh chạy worker như thường>
#
# Vì sao nằm ở worker này chứ không ở 07_score_mix: 07 load checkpoint bằng
# AutoModelForSequenceClassification. Bốn detector zero-shot KHÔNG có checkpoint
# (falcon/gpt-j đóng băng), nên chúng phải được chấm bằng chính score() ở trên. Và PHẢI là
# score() đó, không viết lại — hai đường trong cùng một figure mà dùng hai hàm chấm khác
# nhau thì figure không nói được gì.
#
# Tập mix KHÔNG đi qua vòng split thường: text đã được 06 assemble sẵn nên không melt, và
# không có label nhị phân — chỉ có mix_ratio_actual liên tục. Vòng split thường tính
# AUROC/τ, vô nghĩa ở đây. Nên mix ghi riêng pred_mix*.parquet và KHÔNG vào metrics.json.
#
# Schema đầu ra = ĐÚNG 15 cột KEEP của 07_score_mix + score -> gộp chung được với
# scores_all.parquet mà không lệch cột.
MIX_NAME = "dataset_test_ood_model_mix.parquet"
MIX_KEEP = ["sample_id", "paper_id", "source", "generator_family", "mix_structure",
            "mix_seed", "mix_k_slots", "n_slots", "mix_ratio_actual", "n_seams",
            "word_count", "align_score", "was_mixed", "ai_degree", "in_spec_band"]
RUN_MIX = os.environ.get("RUN_MIX", "0") == "1"
# 0 = FULL (20.358 dòng). Đặt số n để lấy n dòng MỖI Ô (bin x mix_structure) = 20 ô.
MIX_PER_CELL = int(os.environ.get("MIX_PER_CELL", "0"))


def find_mix_file():
    """mix.parquet ở old_dataset/ khi chạy local, ở /kaggle/input khi chạy Kaggle."""
    for base in [Path(KAGGLE_INPUT), Path("/kaggle/input"), Path("."), Path(".."),
                 Path("old_dataset"), Path("output")]:
        try:
            if base.exists():
                hits = sorted(base.rglob(MIX_NAME))
                if hits:
                    return hits[0]
        except OSError:
            continue
    return None


def load_mix():
    p = find_mix_file()
    if p is None:
        log(f"khong thay {MIX_NAME} -> bo qua che do mix")
        return None, None
    df = pd.read_parquet(p)
    have = [c for c in MIX_KEEP if c in df.columns]
    missing = [c for c in MIX_KEEP if c not in df.columns]
    if missing:
        log(f"[!] mix thieu cot {missing} -> de NaN de concat voi 07 khong lech cot")
    out = df[["text"] + have].copy()
    for c in missing:
        out[c] = np.nan
    out = out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)]
    if MIX_PER_CELL > 0:
        _b = (out["mix_ratio_actual"].fillna(0.0) * 10).astype(int).clip(0, 9)
        out = (out.assign(_bin=_b)
                  .groupby(["_bin", "mix_structure"], group_keys=False)
                  .apply(lambda g: g.sample(n=min(len(g), MIX_PER_CELL), random_state=SEED))
                  .drop(columns="_bin"))
        log(f"mix: subsample {MIX_PER_CELL} dong/o (bin x structure)")
    return out.reset_index(drop=True), p


def run_mix():
    import math as _math
    import time as _time

    _rank = int(globals().get("RANK", 0))
    _ws = int(globals().get("WORLD_SIZE", 1))
    df, src = load_mix()
    if df is None or len(df) == 0:
        return
    df["_row_id"] = np.arange(len(df), dtype=np.int64)
    shard = df.iloc[_rank::_ws].reset_index(drop=True) if _ws > 1 else df
    log(f"mix: {len(df)} dong tong -> {len(shard)} dong cua rank {_rank}/{_ws}  <- {src}")
    if SMOKE_TEST:
        _s = score(shard["text"].tolist()[:8])
        log(f"[SMOKE] mix: 8 dong, finite={np.isfinite(_s).all()}, "
            f"min/max={float(np.min(_s)):.4f}/{float(np.max(_s)):.4f}; KHONG luu.")
        return

    ck = OUT_DIR / f"chunks_mix_rank{_rank}"
    ck.mkdir(parents=True, exist_ok=True)
    step = int(globals().get("CHECKPOINT_ROWS", 0) or 200)
    n_ch = _math.ceil(len(shard) / step)
    log(f"mix: {n_ch} chunk x {step} dong, checkpoint tai {ck}")
    t_all = _time.time()
    for k in range(n_ch):
        a, b = k * step, min((k + 1) * step, len(shard))
        p = ck / f"mix_chunk{k:05d}.parquet"
        if p.exists():
            log(f"resume mix chunk {k + 1}/{n_ch}: da co")
            continue
        part = shard.iloc[a:b].copy()
        t0 = _time.time()
        part["score"] = score(part["text"].tolist())
        tmp = p.with_suffix(".tmp.parquet")
        part.drop(columns=["text"]).to_parquet(tmp, index=False)
        os.replace(tmp, p)          # atomic -> het gio session khong de lai chunk viet do dang
        rate = (b - a) / max(_time.time() - t0, 1e-9)
        log(f"mix chunk {k + 1}/{n_ch}: {rate:.2f} mau/s, "
            f"ETA con lai ~{(len(shard) - b) / max(rate, 1e-9) / 3600:.2f}h")

    parts = sorted(ck.glob("mix_chunk*.parquet"))
    if parts:
        allp = pd.concat([pd.read_parquet(x) for x in parts], ignore_index=True)
        dst = OUT_DIR / (f"pred_mix_rank{_rank}.parquet" if _ws > 1 else "pred_mix.parquet")
        allp.to_parquet(dst, index=False)
        log(f"da luu {dst.name}: {len(allp)} dong, wall={_time.time() - t_all:.0f}s")
        log(f"     cot: {list(allp.columns)}")


if RUN_MIX:
    run_mix()

## Chạy trên dual T4 (Kaggle)

`score_worker.py` chạy **một process model-parallel**: `falcon-7b` trên GPU 0, `falcon-7b-instruct`
trên GPU 1 — đúng `DEVICE_1`/`DEVICE_2` của code gốc. **Không dùng `torchrun`**: nhân bản 2 model 7B
lên mỗi T4 sẽ OOM ngay lập tức. Hai forward được launch liên tiếp trước khi chạm vào kết quả nên
2 GPU vẫn chạy **chồng nhau**, không phải nối tiếp.

1. **SMOKE**: tải 2 model, chấm 16 mẫu/lớp mỗi split, in `samples/s` + peak VRAM, **không** lưu
   checkpoint. Đây là bài test VRAM và test NaN thật (fp16).
2. **FULL**: chấm toàn bộ `val`/`test_ood_model`/`test_fp`, checkpoint mỗi `CHECKPOINT_ROWS=100`
   dòng vào `chunks/`. Session hết giờ thì **chạy lại đúng cell này** — chunk hợp lệ được bỏ qua.
3. Cell **gộp chunk** → cell **metrics**.

⚠️ **Yêu cầu**: đúng 2 GPU + **bật Internet**. Lần đầu tải ~29 GB trọng số (~10–25 phút); worker đẩy
cache sang `/kaggle/temp/hf_cache` để không ăn quota output 20 GB. Restart session là mất cache và
phải tải lại — nếu chạy nhiều lần, cân nhắc up 2 model thành Kaggle Dataset.

⏱️ Ước tính FULL: ~8.918 dòng × 2 forward 7B @ 512 token, batch 1 → **~1–2 giờ**. `chunks/` cho phép
cắt ngang bất cứ lúc nào.

🔒 Đổi `OBSERVER_MODEL`/`PERFORMER_MODEL`/`MAX_LEN`/công thức ⇒ `RUN_SIGNATURE` đổi ⇒ chunk cũ bị
**từ chối** kèm thông báo, không âm thầm trộn 2 cấu hình. Muốn chạy lại từ đầu: xoá
`artifacts_binoculars_falcon7b/chunks/`.


In [ ]:
# ── Checkpoint của lần chạy trước: PHẢI xoá khi đổi DATASET_TAG ───────────────
# Chunk/partial cũ mang _data_sha256 của file parquet CŨ. Notebook nào có
# validate_chunk() thì raise; notebook nào không có thì lẳng lặng gộp score của
# dataset cũ vào metric của dataset mới — đúng loại bug im lặng tệ nhất.
#
# Cell này KHÔNG tự xoá. Nó liệt kê ra để bạn nhìn trước, rồi bạn bật cờ.
import shutil
from pathlib import Path

PURGE_STALE = False   # đặt True MỘT LẦN sau khi đổi tag, chạy cell, rồi trả lại False

_base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
_ckpt, _derived = [], []
for _d in sorted(_base.glob("artifacts_*")):
    _ckpt += [p for p in (_d / "chunks",) if p.is_dir()]
    _ckpt += sorted(_d.glob("partial_*.parquet")) + sorted(_d.glob("chunk_*.parquet"))
    _derived += sorted(_d.glob("pred_*.parquet")) + sorted(_d.glob("metrics*.json"))

if not _ckpt and not _derived:
    print("sạch — không có kết quả của lần chạy trước.")
else:
    print(f"CHECKPOINT (score thô, phải xoá): {len(_ckpt)}")
    for _p in _ckpt:
        print("   ", _p)
    print(f"DẪN XUẤT (metric/pred, bị ghi đè khi chạy lại đủ 3 split): {len(_derived)}")
    for _p in _derived:
        print("   ", _p)
    if PURGE_STALE:
        for _p in _ckpt + _derived:
            shutil.rmtree(_p) if _p.is_dir() else _p.unlink()
        print(f"\nĐÃ XOÁ {len(_ckpt) + len(_derived)} mục.")
    else:
        print("\nPURGE_STALE = False -> chưa xoá gì. Bật True rồi chạy lại cell này.")

In [ ]:
# ── SMOKE: kiểm 2 GPU bắt tay + đọc được parquet, vài phút ──────────────────────
# RUN_MIX=1 ở đây để SMOKE kiểm LUÔN việc tìm thấy dataset_test_ood_model_mix.parquet
# và chấm được 8 dòng của nó. Không có nó thì bạn chỉ phát hiện "quên upload mix" SAU
# khi benchmark đã chạy hàng giờ — worker bỏ qua mix im lặng, không báo lỗi.
# Phải thấy 3 dòng sha256/số dòng của deconf2 VÀ một dòng "[SMOKE] mix: 8 dong".
!RUN_MIX=1 SMOKE=1 OMP_NUM_THREADS=2 python3 score_worker.py

In [ ]:
# ── CHẠY FULL: 3 split benchmark + tập MIX trong MỘT lần gọi ────────────────────
# RUN_MIX=1 nằm ở đây thay vì một cell riêng vì worker load model ở module level. Tách
# hai lệnh thì Binoculars load model HAI lần; gộp lại thì load một lần rồi chấm
# splits -> mix liền. Block mix nằm cuối lastde/score_worker.py nên thứ tự tự đúng.
#
# Ước tính: ~1,5h + ~3,7h = ~5,2h. Dưới hạn 12h/session của Kaggle.
# Checkpoint chunk + atomic replace ở CẢ HAI phần -> session chết thì chạy lại cell này
# là tiếp tục đúng chỗ, không chấm lại gì.
# Muốn rút tập mix: thêm MIX_PER_CELL=150 (150 dòng mỗi ô bin x structure = 3.000 dòng).
!RUN_MIX=1 OMP_NUM_THREADS=2 python3 score_worker.py

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

IMPL_VERSION = "paper-exact-falcon-pair-modelparallel-v1-2026-07-28"
OBSERVER_MODEL = "tiiuae/falcon-7b"
PERFORMER_MODEL = "tiiuae/falcon-7b-instruct"
RUN_NAME = "binoculars_falcon7b"
MODEL_DESC = f"{OBSERVER_MODEL} + {PERFORMER_MODEL} (observer/performer)"
FORMULA = "ppl=logPPL_performer(shift1) / xppl=H(p=observer,q=performer,no-shift,all-L); score=-B"
RUN_SIGNATURE = "|".join([IMPL_VERSION, OBSERVER_MODEL, PERFORMER_MODEL, "512",
                          "fp16-weights-fp32-statistics", FORMULA])
TARGET_FPR = 0.01
CHECKPOINT_ROWS = 100

# Nguong toan cuc cua code goc (chi ap dung duoc vi ta khop dung cap Falcon).
# Paper: B < threshold => machine. score = -B => score > -threshold => machine.
PAPER_TAU_ACC = -0.9015310749276843     # BINOCULARS_ACCURACY_THRESHOLD (toi uu F1)
PAPER_TAU_LOWFPR = -0.8536432310785527  # BINOCULARS_FPR_THRESHOLD (toi uu low-FPR)

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_DIR = OUT_DIR / f"artifacts_{RUN_NAME}"
CHUNK_DIR = OUT_DIR / "chunks"

def _merge_chunks(split_name):
    paths = sorted(CHUNK_DIR.glob(f"{split_name}_chunk*.parquet"))
    if not paths:
        raise FileNotFoundError(f"Chua co checkpoint FULL cho split {split_name}.")
    parts = []
    for p in paths:
        part = pd.read_parquet(p)
        required = {"_row_id", "_run_signature", "_data_sha256", "_expected_n", "_chunk_idx",
                    "paper_id", "label", "ai_degree", "generator_family", "score"}
        missing = required.difference(part.columns)
        if missing:
            raise RuntimeError(f"{p.name} thieu cot integrity: {sorted(missing)}")
        if set(part["_run_signature"].unique()) != {RUN_SIGNATURE}:
            raise RuntimeError(f"{p.name} thuoc config/implementation cu; xoa chunks/ va chay lai FULL.")
        if not np.isfinite(part["score"].to_numpy()).all():
            raise RuntimeError(f"{p.name} co score NaN/Inf.")
        parts.append(part)
    expected = {int(x) for part in parts for x in part["_expected_n"].unique()}
    if len(expected) != 1:
        raise RuntimeError(f"Cac chunk {split_name} khong cung expected_n: {expected}")
    expected_n = expected.pop()
    hashes = {str(x) for part in parts for x in part["_data_sha256"].unique()}
    if len(hashes) != 1:
        raise RuntimeError(f"Cac chunk {split_name} den tu nhieu ban dataset: {hashes}")
    merged = pd.concat(parts, ignore_index=True)
    if len(merged) != expected_n:
        raise RuntimeError(f"{split_name}: gop {len(merged)} != expected {expected_n} dong -> "
                           "FULL chua chay xong, chay lai cell FULL (tu resume).")
    if merged["_row_id"].duplicated().any() or set(merged["_row_id"]) != set(range(expected_n)):
        raise RuntimeError(f"{split_name}: _row_id thieu hoac trung giua cac chunk.")
    return merged.sort_values("_row_id").reset_index(drop=True)

val_df = _merge_chunks("val")
ood_df = _merge_chunks("test_ood_model")
fp_df = _merge_chunks("test_fp")
for nm, d in [("val", val_df), ("test_ood_model", ood_df), ("test_fp", fp_df)]:
    print(f"{nm:<16} {len(d):>7,} mau (gop chunk)  label={d['label'].value_counts().to_dict()}")


In [ ]:
# ── Metrics: CUNG cong thuc voi supervised_models/ va adversarial_models/ (tau@FPR=1% tren val) ──
# tpr@fpr*        = operating point tren ROC cua chinh split (metric ranking).
# tpr@tau/fpr@tau = TAU co dinh, calibrate DUY NHAT tren val (metric deployment) -> cot so sanh chinh.
# *_paper_acc / *_paper_lowfpr = nguong toan cuc goc, chi co nghia vi ta khop dung cap Falcon.
def tpr_at_fpr(y, s, fpr):
    y, s = np.asarray(y), np.asarray(s); neg = s[y == 0]
    if len(neg) == 0 or (y == 1).sum() == 0:
        return np.nan, np.nan
    thr = np.quantile(neg, 1 - fpr, method="higher")
    return float((s[y == 1] >= thr).mean()), float(thr)

def auroc(y, s):
    y = np.asarray(y)
    return float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else np.nan

def report_split(name, d):
    if d is None:
        return None
    y, s = d["label"].values, d["score"].values
    row = {"split": name, "n": len(d), "n_ai": int((y == 1).sum()),
           "auroc": auroc(y, s),
           "auprc": float(average_precision_score(y, s)) if len(np.unique(y)) == 2 else np.nan}
    for f in [0.001, 0.01, 0.05]:
        row[f"tpr@fpr{f}"] = tpr_at_fpr(y, s, f)[0]
    row["tpr@tau"] = float((s[y == 1] >= TAU).mean()) if (y == 1).any() else np.nan
    row["fpr@tau"] = float((s[y == 0] >= TAU).mean()) if (y == 0).any() else np.nan
    # nguong goc cua paper: B < t => machine  <=>  score > -t
    for tag, t in [("paper_acc", PAPER_TAU_ACC), ("paper_lowfpr", PAPER_TAU_LOWFPR)]:
        row[f"tpr@{tag}"] = float((s[y == 1] > t).mean()) if (y == 1).any() else np.nan
        row[f"fpr@{tag}"] = float((s[y == 0] > t).mean()) if (y == 0).any() else np.nan
    return row

# ── Suc khoe cua score. KHONG lat dau bang nhan val (statistical_branch_plan.md §4):
#    huong diem da dang ky truoc theo dinh nghia paper (score = -B, B thap = AI).
_std = float(np.std(val_df["score"]))
_auc = auroc(val_df["label"], val_df["score"])
print(f"std(score val) = {_std:.4f} | AUROC(val) = {_auc:.4f} | score_direction=paper_fixed")
print(f"B(val) [min/median/max] = {-val_df['score'].max():.4f} / {-val_df['score'].median():.4f} "
      f"/ {-val_df['score'].min():.4f}   (paper: quanh 0.8-1.0, nguong goc 0.9015)")
if _std < 1e-6:
    print("[!] CANH BAO: score gan nhu hang so - kiem tra lai model/scoring.")
if not np.isnan(_auc) and _auc < 0.5:
    print("[!] CANH BAO: AUROC(val) < 0.5. GIU NGUYEN chieu score theo paper - lat dau bang nhan val "
          "la hau xu ly co nhan, bi cam o statistical_branch_plan.md §4. Di soi lai score_worker.py.")
elif not np.isnan(_auc) and _auc < 0.55:
    print("[!] CANH BAO: AUROC(val) ~ 0.5 - kiem tra lai truoc khi dua vao bang so sanh.")

SMOKE_TEST = False
_, TAU = tpr_at_fpr(val_df["label"], val_df["score"], TARGET_FPR)
print(f"tau (FPR={TARGET_FPR:.0%} tren val) = {TAU:.4f}  (tuong duong B = {-TAU:.4f})")

rows = [r for r in [report_split("val", val_df), report_split("test_ood_model", ood_df),
                     report_split("test_fp", fp_df)] if r]
metrics_df = pd.DataFrame(rows).set_index("split")
pd.set_option("display.width", 250, "display.max_columns", 30)
print(metrics_df.round(4).to_string())

val_auc = metrics_df.loc["val", "auroc"]
ood_auc = metrics_df.loc["test_ood_model", "auroc"]
print("=" * 68); print(f"HEADLINE - {RUN_NAME} (zero-shot, training-free, cap model = paper)"); print("=" * 68)
print(f"1) TPR@FPR=1% OOD-model  : {metrics_df.loc['test_ood_model','tpr@fpr0.01']:.3f}")
print(f"2) FPR thuc @ tau test_fp: {metrics_df.loc['test_fp','fpr@tau']:.4f}  (mong ~{TARGET_FPR})")
print(f"3) AUROC gap val->OOD    : {val_auc:.3f} -> {ood_auc:.3f}  (D={val_auc-ood_auc:+.3f})")
print("   (detector nay khong train nen 'OOD' chi OOD voi 2 nhanh kia, khong phai voi no)")
print(f"4) Zero-config (nguong goc paper 0.9015) tren test_ood_model: "
      f"TPR={metrics_df.loc['test_ood_model','tpr@paper_acc']:.3f}, "
      f"FPR(test_fp)={metrics_df.loc['test_fp','fpr@paper_acc']:.4f}")

if not SMOKE_TEST:
    metrics_out = {"run": RUN_NAME, "model": MODEL_DESC, "method": "binoculars",
                   "implementation": IMPL_VERSION, "eval_mode": "zero_shot", "training_free": True,
                   "backbone_matches_paper": True,
                   "tau": TAU, "tau_source": "val@fpr1pct", "target_fpr": TARGET_FPR,
                   "paper_thresholds": {"accuracy": PAPER_TAU_ACC, "low_fpr": PAPER_TAU_LOWFPR,
                                        "note": "score=-B; B<t => machine"},
                   "score_direction": "paper_fixed_high_is_ai", "sign_flipped_by_val": False,
                   "seed": 42, "n_seeds": 1, "deterministic": True, "sample_weight_used": False,
                   "precision": "fp16_weights_fp32_statistics", "max_len": 512,
                   "gpu_layout": {"cuda:0": "falcon-7b_observer", "cuda:1": "falcon-7b-instruct_performer"},
                   "processes": 1, "gpu_count": 2, "checkpoint_rows": CHECKPOINT_ROWS,
                   "formula": FORMULA,
                   "metrics": metrics_df.reset_index().to_dict("records")}
    (OUT_DIR / "metrics.json").write_text(json.dumps(metrics_out, indent=2, default=float))
    for nm, d in [("val", val_df), ("test_ood_model", ood_df), ("test_fp", fp_df)]:
        d[["paper_id", "label", "ai_degree", "generator_family", "score"]].to_parquet(
            OUT_DIR / f"pred_{nm}.parquet", index=False)
    print("da luu artifacts ->", OUT_DIR)


In [ ]:
# 📊 Tóm tắt nhanh (đọc lại metrics.json vừa lưu)
import json as _json
from pathlib import Path as _Path
_p = _Path(f"artifacts_{RUN_NAME}/metrics.json")
if _p.exists():
    m = _json.loads(_p.read_text())
    print(f"RUN = {m['run']} | model = {m['model']}")
    print(f"backbone khop paper = {m['backbone_matches_paper']} | n_seeds = {m['n_seeds']} "
          f"(deterministic) | lat dau theo val = {m['sign_flipped_by_val']}")
    print(f"tau (val@FPR1%) = {m['tau']:.4f} | formula: {m['formula']}")
    for r in m["metrics"]:
        print(f"{r['split']:16} AUROC={r['auroc']:.4f}  TPR@FPR1%={r['tpr@fpr0.01']:.4f}  "
              f"TPR@FPR0.1%={r['tpr@fpr0.001']:.4f}  FPR@tau={r['fpr@tau']:.4f}  "
              f"| nguong goc paper: TPR={r['tpr@paper_acc']:.4f} FPR={r['fpr@paper_acc']:.4f}")
else:
    print("Chưa có metrics.json — chạy cell FULL RUN trước (bỏ SMOKE=1).")


In [ ]:
# ── CHẨN ĐOÁN "CÓ SO SÁNH ĐƯỢC KHÔNG" — chạy sau cell metric ─────────────────
# Bảng metric ở trên không trả lời được ba câu mà bảng benchmark cuối cùng phải trả lời:
#
#   1. CHIỀU có nhất quán giữa các split? val và test_ood_model cùng phía 0.5 -> không có
#      bug đảo dấu. Nằm hai phía -> không có tín hiệu, và đừng đọc dấu của từng split.
#   2. AUROC có khác ĐOÁN MÒ không? SD của AUROC khi detector vô hiệu là
#      sqrt((n1+n2+1)/(12*n1*n2)) — với val 438+438 thì SD = 0.0195, nên 0.48 và 0.52
#      là CÙNG một kết luận: không phân biệt được.
#   3. TPR@FPR=1% so với mốc nào? Mốc đoán mò là ĐÚNG 0.01 (ROC của classifier vô hiệu là
#      đường chéo), KHÔNG phải 0. Nên 0.0337 là 3.4x đoán mò, không phải "gần như trượt".
#
# In cả mean score theo nhãn: nếu mean(AI) > mean(human) ở CẢ BA split thì hướng đăng ký
# trước là đúng và số thấp là bản chất, không phải lỗi implementation.
import glob
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

_CHANCE = {"tpr@fpr0.001": 0.001, "tpr@fpr0.01": 0.01, "tpr@fpr0.05": 0.05}


def _sd_null(n1, n2):
    return math.sqrt((n1 + n2 + 1) / (12 * n1 * n2))


def _p2(z):
    return math.erfc(abs(z) / math.sqrt(2))


_preds = sorted(p for p in glob.glob("**/pred_*.parquet", recursive=True)
                if "pred_matched" not in p)
if not _preds:
    print("khong tim thay pred_*.parquet -> chay cell metric truoc")
else:
    print("=" * 92)
    print("CHIEU SCORE THEO NHAN (mean; huong dang ky: cao = AI)")
    print("=" * 92)
    print(f"  {'split':18}{'n':>7}{'mean human':>13}{'mean AI':>12}{'gap':>11}  dung chieu?")
    _signs = {}
    for _p in _preds:
        _d = pd.read_parquet(_p).dropna(subset=["score"])
        _sp = Path(_p).stem.replace("pred_", "")
        _h = _d.loc[_d["label"] == 0, "score"]
        _a = _d.loc[_d["label"] == 1, "score"]
        if len(_a) == 0:
            print(f"  {_sp:18}{len(_d):>7}{_h.mean():13.4f}{'—':>12}{'—':>11}  (1 lop, bo qua)")
            continue
        _gap = _a.mean() - _h.mean()
        _signs[_sp] = _gap
        print(f"  {_sp:18}{len(_d):>7}{_h.mean():13.4f}{_a.mean():12.4f}{_gap:+11.4f}"
              f"  {'CO' if _gap > 0 else 'KHONG'}")
    if len(_signs) >= 2:
        _ok = all(v > 0 for v in _signs.values()) or all(v < 0 for v in _signs.values())
        print(f"\n  -> chieu {'NHAT QUAN' if _ok else 'KHONG nhat quan'} qua "
              f"{len(_signs)} split"
              + ("" if _ok else "  <-- dau hieu KHONG CO TIN HIEU, dung doc dau tung split"))
        if _ok and all(v > 0 for v in _signs.values()):
            print("     dung huong dang ky truoc => implementation dung chieu; so thap la ban chat")

    print()
    print("=" * 92)
    print("CO HON DOAN MO KHONG")
    print("=" * 92)
    print(f"  {'split':18}{'AUROC':>9}{'SD_null':>10}{'z':>8}{'p':>9}  ket luan")
    for _p in _preds:
        _d = pd.read_parquet(_p).dropna(subset=["score"])
        _sp = Path(_p).stem.replace("pred_", "")
        _n1 = int((_d["label"] == 1).sum())
        _n0 = int((_d["label"] == 0).sum())
        if _n1 == 0 or _n0 == 0:
            continue
        _a = float(roc_auc_score(_d["label"], _d["score"]))
        _sd = _sd_null(_n0, _n1)
        _z = (_a - 0.5) / _sd
        _v = "hon ngau nhien" if (_p2(_z) < 0.05 and _z > 0) else (
            "DUOI ngau nhien" if _p2(_z) < 0.05 else "KHONG phan biet duoc voi doan mo")
        print(f"  {_sp:18}{_a:9.4f}{_sd:10.4f}{_z:+8.2f}{_p2(_z):9.4f}  {_v}")

    print()
    print("=" * 92)
    print("TPR SO MOC DOAN MO (moc = dung bang FPR, khong phai 0)")
    print("=" * 92)
    _art = Path(_preds[0]).parent
    _mp = _art / "metrics.json"
    _m = json.loads(_mp.read_text()) if _mp.exists() else None
    if _m is None:
        print("  khong doc duoc metrics.json")
    else:
        print(f"  {'split':18}{'chi so':16}{'gia tri':>10}{'doan mo':>10}{'lan':>7}{'z':>8}")
        for _r in _m["metrics"]:
            _npos = _r.get("n_ai") or 0
            _nneg = (_r.get("n") or 0) - _npos
            if not _npos or not _nneg:
                continue
            for _k, _c in _CHANCE.items():
                _v = _r.get(_k)
                if _v is None or (isinstance(_v, float) and math.isnan(_v)):
                    continue
                _z = (_v - _c) / math.sqrt(_c * (1 - _c) / _npos)
                # nguong FPR duoc dat boi phan vi tren NEGATIVE -> so mau quyet dinh = _c*_nneg
                _ndec = _c * _nneg
                _flag = f"  <-- nguong do ~{_ndec:.1f} mau quyet dinh, DUNG BAO" if _ndec < 3 else ""
                print(f"  {_r['split']:18}{_k:16}{_v:10.4f}{_c:10.4f}{_v / _c:6.1f}x{_z:+8.1f}{_flag}")